## 1. Setup

In [2]:
from pathlib import Path

In [3]:
url = "https://opendata.schleswig-holstein.de/dataset/5e0652d8-7f59-42fc-92ab-a6358f800e1d/resource/4cea9aa9-b47f-4e80-ad71-514d3783650c/download/fahrplandaten"
out_dir = Path("../data/fahrplandaten/")

# 2. Download Data

In [ ]:
import requests
from io import BytesIO
from zipfile import ZipFile
import os

In [ ]:
# Create out directory
os.makedirs(out_dir, exist_ok=True)

# Download and unpack
response = requests.get(url)
response.raise_for_status()

# Unpack response
with ZipFile(BytesIO(response.content)) as zip_ref:
    zip_ref.extractall(out_dir)

print(f"Extracted to {out_dir}.")

del response
del zip_ref

# 3. Query Data

In [42]:
import pandas as pd

In [ ]:
# Collect all dataframes
def load_df(name: str, verbose=True) -> pd.DataFrame:
    df = pd.read_csv(out_dir / f"{name}.txt")
    if verbose:
        print(f"df_{name}:\t{df.columns.to_list()}")
    return df

df_agency = load_df("agency")
df_calendar = load_df("calendar")
df_calendar_dates = load_df("calendar_dates")
# df_frequencies = load_df("frequencies")
df_routes = load_df("routes")
# df_service_alerts = load_df("service_alerts")
df_shapes = load_df("shapes")
df_stop_times = load_df("stop_times")
df_stops = load_df("stops")
df_transfers = load_df("transfers")
df_trips = load_df("trips")

df_agency:	['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone']
df_calendar:	['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']
df_calendar_dates:	['service_id', 'date', 'exception_type']
df_routes:	['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type', 'route_color', 'route_text_color', 'route_desc']
df_shapes:	['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence']
df_stop_times:	['trip_id', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'stop_headsign', 'arrival_time', 'departure_time']
df_stops:	['stop_id', 'stop_code', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'location_type', 'parent_station', 'wheelchair_boarding', 'platform_code', 'zone_id']
df_transfers:	['from_stop_id', 'to_stop_id', 'transfer_type', 'min_transfer_time', 'from_route_id', 'to_route_id', 'from_trip_id', 'to_trip_id']
df_trips:	['route_id', 'service_id

/tmp/ipykernel_334551/1544208904.py:3: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out_dir / f"{name}.txt")


In [ ]:
""" Filter dataframes for KVG data """

# Get KVG agency id
kvg_id = df_agency[df_agency.agency_name == "Kieler Verkehrsgesellschaft mbH"].agency_id.iloc[0]
# print(kvg_id)

# Filter routes to kvg-only
df_kvg_routes = df_routes[df_routes.agency_id == kvg_id].loc[:, ["route_id", "route_short_name"]]
# print(df_kvg_routes)

# Filter trips to kvg-only
df_kvg_trips = df_trips[df_trips.route_id.isin(df_kvg_routes.route_id)].loc[:, ["route_id", "trip_id", "trip_headsign", "service_id"]]

# Filter calendar dates
# df_kvg_calendar = df_calendar[df_calendar.service_id.isin(df_kvg_trips.service_id)]  # every day entry is 0
df_kvg_calendar_dates = df_calendar_dates[df_calendar_dates.service_id.isin(df_kvg_trips.service_id)].loc[:, ["service_id", "date"]]
df_kvg_calendar_dates.date = pd.to_datetime(df_kvg_calendar_dates.loc[:, "date"], format="%Y%m%d")

# Filter stop times to kvg-only
df_kvg_stop_times = df_stop_times[df_stop_times.trip_id.isin(df_kvg_trips.trip_id)].loc[:, ["trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"]]

# Filter stops by kvg-only
df_kvg_stops = df_stops[df_stops.stop_id.isin(pd.unique(df_kvg_stop_times.stop_id))].loc[:, ["stop_id", "stop_name", "stop_lat", "stop_lon"]]


          trip_id         stop_id  stop_sequence arrival_time departure_time
374372  493696120  de:01002:49076             20     24:00:00       24:05:00
374412  493696124  de:01002:49076             20     25:00:00       25:05:00
374452  493696123  de:01002:49076             20     26:00:00       26:05:00
374492  493696122  de:01002:49076             20     27:00:00       27:05:00
374532  493696121  de:01002:49076             20     27:55:00       28:00:00
...           ...             ...            ...          ...            ...
622518  511881476  de:01002:49076             17     21:45:00       21:50:00
622551  511881475  de:01002:49076             17     22:15:00       22:20:00
622584  511881480  de:01002:49076             17     22:45:00       22:50:00
622617  511881479  de:01002:49076             17     23:15:00       23:20:00
622650  511881478  de:01002:49076             17     23:45:00       23:50:00

[5610 rows x 5 columns]


In [45]:
""" Enhance data """

# Arrival and departure times to timedeltas
df_kvg_stop_times.arrival_time = pd.to_timedelta(df_kvg_stop_times.arrival_time)
df_kvg_stop_times.departure_time = pd.to_timedelta(df_kvg_stop_times.departure_time)

# Add dates to stop times
df_kvg_dates = df_kvg_trips.merge(df_calendar_dates, on="service_id").loc[:, ["trip_id", "date"]]
df_kvg_dates.date = pd.to_datetime(df_kvg_dates.date, format="%Y%m%d")
df_kvg_stop_times = df_kvg_stop_times.merge(df_kvg_dates, on="trip_id")

# Combine date and time
df_kvg_stop_times.arrival_time = df_kvg_stop_times.date + df_kvg_stop_times.arrival_time
df_kvg_stop_times.departure_time = df_kvg_stop_times.date + df_kvg_stop_times.departure_time

In [46]:
""" Combine dataframes """

df_kvg = df_kvg_stop_times \
    .merge(df_kvg_stops, on="stop_id") \
    .merge(df_kvg_trips, on="trip_id") \
    .merge(df_kvg_routes, on="route_id") \
    .loc[:, ["route_short_name", "stop_sequence", "arrival_time", "departure_time", "stop_name", "stop_lon", "stop_lat"]]

"""
# routes.route_short_name   stop_times.stop_sequence    stop_times.arrival_time stop_times.departure_time   stops.stop_name stops.stop_lon  stops.stop_lat  
# 91                        0..n                        2026-01-01 12:00:00     2026-01-01 12:05_00         Hauptbahnhof    52.96           11.08           
"""
df_kvg

,route_short_name,stop_sequence,arrival_time,departure_time,stop_name,stop_lon,stop_lat
0,X91,0,2026-09-05 10:52:00,2026-09-05 10:52:00,Kiel Hauptbahnhof,10.130686,54.315500
1,X91,0,2026-09-19 10:52:00,2026-09-19 10:52:00,Kiel Hauptbahnhof,10.130686,54.315500
2,X91,1,2026-09-05 10:57:00,2026-09-05 10:57:00,Kiel Wilhelmplatz,10.120359,54.322625
3,X91,1,2026-09-19 10:57:00,2026-09-19 10:57:00,Kiel Wilhelmplatz,10.120359,54.322625
4,X91,2,2026-09-05 10:59:00,2026-09-05 10:59:00,Kiel Gravelottestraße,10.113324,54.325003
...,...,...,...,...,...,...,...
8752133,11,32,2026-09-14 00:09:00,2026-09-14 00:09:00,Kiel Wik Kanal,10.131386,54.365402
8752134,11,32,2026-09-21 00:09:00,2026-09-21 00:09:00,Kiel Wik Kanal,10.131386,54.365402
8752135,11,32,2026-09-28 00:09:00,2026-09-28 00:09:00,Kiel Wik Kanal,10.131386,54.365402
8752136,11,32,2026-10-05 00:09:00,2026-10-05 00:09:00,Kiel Wik Kanal,10.131386,54.365402


In [47]:
""" Filter data """
start_date = "2026-11-05"
end_date = "2026-11-06"
df_kvg = df_kvg[(df_kvg.departure_time > start_date) & (df_kvg.arrival_time < end_date)]
df_kvg.sort_values(by="arrival_time")

,route_short_name,stop_sequence,arrival_time,departure_time,stop_name,stop_lon,stop_lat
272199,N14,18,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
214360,N22,21,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
35399,N62,18,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
367299,N12,37,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
317338,N14,35,2026-11-05 00:00:00,2026-11-05 00:05:00,Kiel Hauptbahnhof,10.130686,54.315500
...,...,...,...,...,...,...,...
5501213,31,33,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel Isarweg,10.193354,54.289073
6911117,14,35,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel Hasselfelde,10.186668,54.343252
3315125,51,13,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel Alsenstraße,10.129716,54.334719
584029,91,24,2026-11-05 23:59:00,2026-11-05 23:59:00,Kiel Hasselrade,10.092878,54.319781


# 4. Save data

In [58]:
import os

output_file = "../data/kvg_data.csv"


os.makedirs(os.path.dirname(output_file), exist_ok=True)
df_kvg.to_csv(output_file, index=False)